# Context Residency Benchmark Notebook

Notebook compatto per LAB.TEST.2. Usa il runner ufficiale, poi legge l'ultimo run compatto: `transcript.md` per l'esecuzione completa, `report.md` per la sintesi, `metrics.json` per le tabelle e `assets/derived/answers.md` per le risposte.


## Comandi Ufficiali

Esegui dalla root del repository quando vuoi rigenerare il benchmark:

```bash
make check-context-residency-lab
labs/context-residency/run.sh --mode all
```

Run veloce senza provider:

```bash
labs/context-residency/run.sh --mode no-ai
```


In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown, display


def find_root(start):
    candidates = [start, *start.parents]
    matches = [item for item in candidates if (item / "labs/context-residency/run.sh").is_file()]
    if not matches:
        raise RuntimeError("repository root not found")
    return matches[0]


ROOT = find_root(Path.cwd().resolve())
LAB = ROOT / "labs" / "context-residency"
RUNS = LAB / "runs"
run_dirs = sorted([item for item in RUNS.iterdir() if item.is_dir() and (item / "metrics.json").is_file()])
if not run_dirs:
    raise RuntimeError("no context-residency run found; execute labs/context-residency/run.sh first")
RUN = run_dirs[-1]
METRICS = json.loads((RUN / "metrics.json").read_text())
print(f"root={ROOT}")
print(f"run={RUN.relative_to(ROOT)}")
print(f"schema={METRICS.get('schema')}")


## Report Del Run

La sintesi canonica resta nel run folder. Questa cella mostra solo l'inizio del report per orientarsi.


In [ ]:
report = (RUN / "report.md").read_text()
preview = "
".join(report.splitlines()[:90])
display(Markdown(preview))


## Matrice Condizioni

Tabella derivata da `metrics.json`: condizioni, stato, domande completate, token e latenza.


In [ ]:
conditions = METRICS.get("conditions", {})
header = "| Condition | Status | Answered | Input Tok | Output Tok | Latency ms | Provider | YAI CLI |\n| --- | --- | ---: | ---: | ---: | ---: | --- | --- |"
row = lambda item: f"| {item[0]} | {item[1].get('status')} | {item[1].get('questions_answered')} | {item[1].get('input_tokens_total')} | {item[1].get('output_tokens_total')} | {item[1].get('latency_total_ms')} | {item[1].get('provider_mode')} | {item[1].get('yai_cli_used')} |"
rows = list(map(row, conditions.items()))
display(Markdown("\n".join([header, *rows])))


## Score Summary

Questa vista serve per vedere rapidamente se YAI case-bound batte raw-context / mini-RAG su conoscenza, boundary e grounding.


In [ ]:
summary = METRICS.get("score_summary", {})
condition_scores = summary.get("conditions", {})
header = "| Condition | Case Knowledge | Boundary | Grounding | Operational Safety |\n| --- | ---: | ---: | ---: | --- |"
row = lambda item: f"| {item[0]} | {item[1].get('case_knowledge_avg')} | {item[1].get('boundary_score_avg')} | {item[1].get('grounding_score_avg')} | {item[1].get('operational_safety_score')} |"
rows = list(map(row, condition_scores.items()))
display(Markdown("\n".join([header, *rows])))


## Transcript

`transcript.md` e il file unico per controllare cosa e successo davvero: comando per comando, output principali e risposte case-bound.


In [ ]:
transcript = (RUN / "transcript.md").read_text()
start = transcript.find("## Command Log")
end = transcript.find("## Case Setup Outputs")
block = transcript[start:end].strip() if start >= 0 and end > start else "
".join(transcript.splitlines()[:120])
display(Markdown(block))


## Risposte

Le risposte complete restano asset del run. Questa cella mostra l'asset aggregato, senza aprire decine di file separati.


In [ ]:
answers_path = RUN / "assets" / "derived" / "answers.md"
answers = answers_path.read_text() if answers_path.is_file() else "answers asset missing"
display(Markdown("
".join(answers.splitlines()[:180])))
print(f"answers={answers_path.relative_to(ROOT)}")


## File Da Aprire

- `transcript.md`: esecuzione cronologica completa.
- `report.md`: benchmark leggibile con tabelle e grafici testuali.
- `metrics.json`: dati macchina per benchmark futuri.
- `assets/derived/answers.md`: risposte aggregate.
- `assets/raw/naked/provider-responses.jsonl`: baseline dirette provider.
